In [11]:
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import random
from faker import Faker

# paths tools
import os
import sys

# making src visible from here
src_root = os.path.abspath("..")
if src_root not in  sys.path:
    sys.path.append(src_root)

# custom
import src.utils as utils
import src.models as models

# imports re for text cleaning
import re
from datetime import datetime, timedelta, date

# we will ignore pandas warning
import warnings
warnings.filterwarnings('ignore')

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
# i recommend setting num_transactions=50000 in case of cloud computations
# for larger sample size
# adapt the num_inns and num_groups to your needs
df, inns, inn2group, group2name = utils.make_interactions_dataset(
    num_transactions=1000,
    num_inns=80,
    num_groups=10
)

df_train, df_test = train_test_split(df, test_size=0.2)

# dropping duplicate interactions from test which are present in train
train_kt_dt_set = set(zip(df_train['inn_kt'], df_train['inn_dt']))
df_test = df_test[~df_test[['inn_kt', 'inn_dt']].apply(tuple, axis=1).isin(train_kt_dt_set)]
df_test = df_test.reset_index(drop=True)

df.head()

,id_trans,inn_kt,inn_dt,c_sum,date,nazn,kt_group_num,dt_group_num,kt_group_name,dt_group_name,raw_word,word
0,1,1206200495,3240170234,191994.13,2021-07-06,колесо,5,5,remain,remain,word2,clean_2
1,2,4853996563,6063478943,189831.76,2022-02-05,чай,8,8,pretty,pretty,None,None
2,3,6884336871,9614808274,129945.91,2023-12-02,кофе,9,5,or,remain,word3,clean_3
3,4,2598461971,4843048064,159428.24,2021-03-27,овощи,8,8,pretty,pretty,word3,None
4,5,5072157107,3116030783,52003.66,2022-12-01,хлеб,9,2,or,among,word4,clean_4


Импортнём коллейт чтобы проверить в боевых условиях. Протестим как заводится модель:

In [19]:
inn2id = {
    "4853996563": torch.tensor(1),
    "6063478943": torch.tensor(2),
    "1206200495": torch.tensor(3),
    "3240170234": torch.tensor(4),
}

word2id = {
    "pretty": torch.tensor(1),
    "or": torch.tensor(2),
    "remain": torch.tensor(3),
}

item_1 = {
    "kt_features": {
        "inn_kt": inn2id["1206200495"],
        "kt_group_name": word2id["remain"],
    },
    "dt_features": {
        "inn_dt": inn2id["3240170234"],
        "dt_group_name": word2id["pretty"],
    }
}


item_2 = {
    "kt_features": {
        "inn_kt": inn2id["4853996563"],
        "kt_group_name": word2id["pretty"],
    },
    "dt_features": {
        "inn_dt": inn2id["6063478943"],
        "dt_group_name": word2id["or"],
    }
}

dataset_items = [item_1, item_2]
batch = utils.collate_fn(dataset_items)

In [23]:
model = models.DeepFM(
    embedding_dim=2,
    num_heads=1,
    kt_features={"inn_kt": 4, "kt_group_name": 3},
    dt_features={"inn_dt": 4, "dt_group_name": 3},
)

DeepFM(
  (embeddings_kt): ModuleDict(
    (inn_kt): Embedding(4, 2)
    (kt_group_name): Embedding(3, 2)
  )
  (embeddings_dt): ModuleDict(
    (inn_dt): Embedding(4, 2)
    (dt_group_name): Embedding(3, 2)
  )
  (attention_kt): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=2, out_features=2, bias=True)
  )
  (attention_dt): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=2, out_features=2, bias=True)
  )
  (weights_kt): Linear(in_features=2, out_features=1, bias=True)
  (weights_dt): Linear(in_features=2, out_features=1, bias=True)
)